# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors (FAIR^2)

This notebook demonstrates how to explore the FAIR^2 dataset using the [mlcroissant](https://github.com/mlcommons/croissant) library, following the Croissant data ecosystem standard.

### Dataset Source
The dataset is described by a Croissant schema at:

**https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json**

In [ ]:
# Ensure `mlcroissant` is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'
# Load the Croissant schema
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs (using the `@id` for every entity).

Let's enumerate all the record sets in this dataset, display their `@id` and their fields' `@id`, and examine a preview from the first record set.

In [ ]:
# Discover all record sets and their fields

record_sets = dataset.record_sets

if not record_sets:
    print("No record sets found in this dataset.\nPlease check if the schema provides any tabular data.")
else:
    print(f"\nFound {len(record_sets)} record set(s):\n")
    for rs in record_sets:
        print(f"Record set @id: {rs['@id']}")
        fields = rs.get('field', [])
        # If only one field, will be dict, else list
        if isinstance(fields, dict):
            fields = [fields]
        print("  Fields:")
        for f in fields:
            print(f"    - {f['@id']} ({f.get('name', '')})")
        print()
    # Show a data preview for the first record set
    first_rs_id = record_sets[0]['@id']
    preview_records = list(dataset.records(record_set=first_rs_id))
    print(f"\nPreview of first 2 records from record set '{first_rs_id}':")
    for r in preview_records[:2]:
        print(r)
    # Store important variables for later
    record_set_ids = [rs['@id'] for rs in record_sets]
    # For later use
    first_record_set_id = record_set_ids[0]

## 3. Data Extraction
Load all records for each record set into a pandas DataFrame for analysis. Each record set is addressed by its full Croissant `@id`. Columns (fields) are always referenced by their `@id`.

In [ ]:
# Collect all dataframes for all record sets, by @id
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Display columns (fields' @id) of the main record set
main_rs_id = first_record_set_id
print(f"Columns (fields by @id) in record set '{main_rs_id}':\n{dataframes[main_rs_id].columns.tolist()}")
dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)

Let's perform some common data processing steps. (1) We'll filter records on a selected numeric field, (2) normalize it, and (3) group by a selected categorical field. All references are done via the fields' `@id`.

You'll want to adapt `numeric_field_id` and `group_field_id` to your actual dataset using the list printed above. We'll guess at some likely IDs:

In [ ]:
# Set these IDs based on your data dictionary; here we use likely ones:
# Example: 'https://api.app.sen.science/frontiers/7862866/field_age' (replace with real one from fields list)

--
# You should replace the following with actual field @ids from your dataset overview.
# For illustration, let's create dynamic selectors based on field names (as hints)

main_df = dataframes[main_rs_id]

# List all columns for user reference
print("Column IDs and inferred names in main record set:")
for col in main_df.columns:
    print(f"  {col}")

# Try to pick out an age-like numeric field
import re
possible_numeric = [col for col in main_df.columns if re.search(r'age|interval|duration|number|year', col, re.IGNORECASE)]
if possible_numeric:
    numeric_field_id = possible_numeric[0]
    print(f"Selected field for numeric EDA: {numeric_field_id}")
else:
    numeric_field_id = main_df.columns[0]  # fallback

# Try to pick a grouping/categorical field (sex, site, MSI status, ...)
possible_categorical = [col for col in main_df.columns if re.search(r'sex|gender|site|location|status|type|category|phenotype', col, re.IGNORECASE)]
group_field_id = possible_categorical[0] if possible_categorical else main_df.columns[1] if len(main_df.columns)>1 else main_df.columns[0]

# Make sure numeric field is numeric, if possible
main_df[numeric_field_id] = pd.to_numeric(main_df[numeric_field_id], errors='coerce')
# Remove nulls
filtered_df = main_df[main_df[numeric_field_id].notnull()]
threshold = filtered_df[numeric_field_id].mean()  # Use mean as example threshold
above_thresh = filtered_df[filtered_df[numeric_field_id] > threshold]

print(f"\nFiltered records with {numeric_field_id} > {threshold:.2f} (mean value):\n")
print(above_thresh.head())

# Normalize the numeric field among those filtered
norm_col = f"{numeric_field_id}_normalized"
above_thresh[norm_col] = (above_thresh[numeric_field_id] - above_thresh[numeric_field_id].mean()) / above_thresh[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id}:\n")
print(above_thresh[[numeric_field_id, norm_col]].head())

if group_field_id in above_thresh.columns:
    grouped = above_thresh.groupby(group_field_id)[numeric_field_id].mean().to_frame('mean')
    print(f"\nGrouped data by {group_field_id} (mean {numeric_field_id}):\n")
    print(grouped.head())

## 5. Visualization
Below, we'll create a plot to show the distribution of our selected numeric field and compare means by the selected group (`@id`).

In [ ]:
# Plotting the numeric field's distribution and its mean per group
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 4))
sns.histplot(above_thresh[numeric_field_id], bins=15, kde=True)
plt.xlabel(numeric_field_id)
plt.title(f"Distribution of {numeric_field_id}")
plt.show()

# If group_field_id is present, plot means by group
if group_field_id in above_thresh.columns:
    plt.figure(figsize=(8,4))
    means = above_thresh.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    sns.barplot(x=group_field_id, y=numeric_field_id, data=means)
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

* In this notebook, you've learned how to access and explore a FAIR-compliant dataset using the Croissant schema and the `mlcroissant` Python library.
* All references to entities (record sets, fields/columns) were based on their `@id`.
* This workflow can be adapted for any other Croissant dataset, facilitating reproducible, robust data science in biomedical and clinical informatics.